# Phase 2, kFold Cross-Validation (5 folds × 3 repeats)
---
All 39 shape parameters. RepeatedKFold evaluation.
All models regularised via GridSearchCV inside a Pipeline.
Metrics: R², MSE, MAE, clinical tolerance ±0.5 and ±1.0 BCS.

In [1]:
import warnings

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin
from mord import LogisticAT

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_excel('COMBINED_HORSES_DATA.xlsx')

beta_cols = [f'beta_{i}' for i in range(0, 39)]
X = df[beta_cols]
y = df['BCS']
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")

Dataset: 208 samples, 39 features


In [3]:
# sklearn-compatible wrapper for mord ordinal logistic regression
class OrdinalWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    # fit the ordinal model after rounding y to integers
    def fit(self, X, y):
        y_int = np.round(y).astype(int)
        self.model_ = LogisticAT(alpha=self.alpha)
        self.model_.fit(X, y_int)
        return self

    # return predictions cast to float
    def predict(self, X):
        return self.model_.predict(X).astype(float)

## Evaluation function
---
GridSearchCV runs inside a Pipeline so the scaler is re-fit per fold.
The best model is then evaluated with a fresh RepeatedKFold loop,
creating a new model instance per fold to avoid any state leakage.

In [4]:
# tune a model with GridSearchCV then evaluate it with RepeatedKFold and print metrics
def evaluate_model(model, param_grid, X, y, name, cv_folds=5, n_repeats=3):
    cv = RepeatedKFold(n_splits=cv_folds, n_repeats=n_repeats, random_state=42)

    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])

    gs = GridSearchCV(pipe, param_grid if param_grid else {},
                      cv=cv, scoring='neg_mean_squared_error',
                      n_jobs=-1, verbose=0)
    gs.fit(X, y)
    best = gs.best_estimator_

    # extract best params for fresh instantiation per fold
    step = best.named_steps['model']
    clean_params = step.get_params()

    test_r2, test_mse, test_mae = [], [], []
    all_true, all_pred = [], []

    for train_idx, test_idx in cv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        m = step.__class__(**clean_params)
        m.fit(X_tr_s, y_tr)
        preds = m.predict(X_te_s)

        test_r2.append(r2_score(y_te, preds))
        test_mse.append(mean_squared_error(y_te, preds))
        test_mae.append(mean_absolute_error(y_te, preds))
        all_true.extend(y_te)
        all_pred.extend(preds)

    errors = np.abs(np.array(all_true) - np.array(all_pred))

    print(f"{name}")
    if gs.best_params_:
        print(f"  Best Params: {gs.best_params_}")
    print(f"  Test R²:  {np.mean(test_r2):.4f}")
    print(f"  Test MSE: {np.mean(test_mse):.4f}")
    print(f"  Test MAE: {np.mean(test_mae):.4f}")
    print(f"  ±0.5 BCS: {np.mean(errors <= 0.5):.1%}")
    print(f"  ±1.0 BCS: {np.mean(errors <= 1.0):.1%}")
    print()
    return best

## Train & evaluate all models
---

In [5]:
print("Evaluating Models, Repeated KFold (5 Folds × 3 Repeats)")
print()

best_ols = evaluate_model(LinearRegression(), {}, X, y, "OLS")

best_lasso = evaluate_model(
    Lasso(random_state=42),
    {'model__alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]},
    X, y, "Lasso"
)

best_ridge = evaluate_model(
    Ridge(random_state=42),
    {'model__alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0],
     'model__solver': ['auto', 'svd', 'cholesky', 'lsqr']},
    X, y, "Ridge"
)

best_pls = evaluate_model(
    PLSRegression(),
    {'model__n_components': [2, 3, 4, 5, 6, 7, 8, 10, 12]},
    X, y, "PLS"
)

best_rf = evaluate_model(
    RandomForestRegressor(random_state=42),
    {'model__n_estimators': [100, 200, 300],
     'model__max_depth': [3, 5, 7, 10],
     'model__min_samples_split': [2, 5, 10],
     'model__min_samples_leaf': [1, 2, 4]},
    X, y, "Random Forest"
)

best_mlp = evaluate_model(
    MLPRegressor(max_iter=2000, random_state=42),
    {'model__hidden_layer_sizes': [(4,), (4, 4), (8,)],
     'model__activation': ['relu', 'tanh'],
     'model__alpha': [0.001, 0.01, 0.1, 1.0, 5.0],
     'model__solver': ['adam', 'lbfgs']},
    X, y, "MLP"
)

best_ordinal = evaluate_model(
    OrdinalWrapper(),
    {'model__alpha': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]},
    X, y, "Ordinal Regression (Regularised)"
)

Evaluating Models, Repeated KFold (5 Folds × 3 Repeats)

OLS
  Test R²:  0.2275
  Test MSE: 0.5697
  Test MAE: 0.5991
  ±0.5 BCS: 50.8%
  ±1.0 BCS: 79.6%

Lasso
  Best Params: {'model__alpha': 0.01}
  Test R²:  0.2984
  Test MSE: 0.5176
  Test MAE: 0.5665
  ±0.5 BCS: 52.4%
  ±1.0 BCS: 81.6%

Ridge
  Best Params: {'model__alpha': 50.0, 'model__solver': 'lsqr'}
  Test R²:  0.3410
  Test MSE: 0.4872
  Test MAE: 0.5499
  ±0.5 BCS: 55.4%
  ±1.0 BCS: 83.3%

PLS
  Best Params: {'model__n_components': 4}
  Test R²:  0.3165
  Test MSE: 0.5057
  Test MAE: 0.5617
  ±0.5 BCS: 54.3%
  ±1.0 BCS: 82.5%

Random Forest
  Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}
  Test R²:  0.2427
  Test MSE: 0.5596
  Test MAE: 0.5872
  ±0.5 BCS: 54.3%
  ±1.0 BCS: 81.9%

MLP
  Best Params: {'model__activation': 'tanh', 'model__alpha': 5.0, 'model__hidden_layer_sizes': (4,), 'model__solver': 'adam'}
  Test R²:  0.2081
  Test MSE: 0.5881

## VotingRegressor Ensemble
---
Ordinal Regression is excluded: integer-class output is
incompatible with continuous averaging in VotingRegressor.

In [7]:
print("Voting Regressor (OLS + Lasso + Ridge + PLS + RF + MLP)")
print()

cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
test_r2, test_mse, test_mae = [], [], []
all_true, all_pred = [], []

# extract a fresh model instance from a fitted pipeline
def _fresh(pipe):
    step = pipe.named_steps['model']
    return step.__class__(**step.get_params())

for train_idx, test_idx in cv.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    ens = VotingRegressor(estimators=[
        ('ols',   _fresh(best_ols)),
        ('lasso', _fresh(best_lasso)),
        ('ridge', _fresh(best_ridge)),
        ('pls',   _fresh(best_pls)),
        ('rf',    _fresh(best_rf)),
        ('mlp',   _fresh(best_mlp)),
    ])
    ens.fit(X_tr_s, y_tr)
    preds = ens.predict(X_te_s)

    test_r2.append(r2_score(y_te, preds))
    test_mse.append(mean_squared_error(y_te, preds))
    test_mae.append(mean_absolute_error(y_te, preds))
    all_true.extend(y_te)
    all_pred.extend(preds)

errors = np.abs(np.array(all_true) - np.array(all_pred))
print(f"  Test R²:  {np.mean(test_r2):.4f}")
print(f"  Test MSE: {np.mean(test_mse):.4f}")
print(f"  Test MAE: {np.mean(test_mae):.4f}")
print(f"  ±0.5 BCS: {np.mean(errors <= 0.5):.1%}")
print(f"  ±1.0 BCS: {np.mean(errors <= 1.0):.1%}")

Voting Regressor (OLS + Lasso + Ridge + PLS + RF + MLP)

  Test R²:  0.3288
  Test MSE: 0.4959
  Test MAE: 0.5539
  ±0.5 BCS: 53.5%
  ±1.0 BCS: 83.0%
